In [ ]:
# DDL / Spark SQL Views

Dieses Notebook lädt die bereinigten Parquet-Daten aus dem HDFS und registriert sie als Spark SQL View.

Ziel:
- processed Parquet-Daten lesen
- Schema prüfen
- Spark SQL View `parking_violations` erstellen
- erste SQL-Testqueries ausführen

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_DDL") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memory", "8g") \
    .config("spark.cores.max", "4") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/09 20:43:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/09 20:43:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned"

df = spark.read.parquet(processed_path)

df.printSchema()

26/05/09 20:44:07 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


root
 |-- summons_number: string (nullable = true)
 |-- plate_id: string (nullable = true)
 |-- registration_state: string (nullable = true)
 |-- issue_date: string (nullable = true)
 |-- violation_time: string (nullable = true)
 |-- violation_county: string (nullable = true)
 |-- violation_precinct: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- vehicle_make: string (nullable = true)
 |-- vehicle_body_type: string (nullable = true)
 |-- violation_code: string (nullable = true)
 |-- violation_description: string (nullable = true)
 |-- issue_date_parsed: date (nullable = true)
 |-- issue_year: integer (nullable = true)
 |-- issue_month: integer (nullable = true)
 |-- issue_weekday: integer (nullable = true)
 |-- fiscal_year: integer (nullable = true)



In [3]:
df.createOrReplaceTempView("parking_violations")

spark.sql("""
SELECT *
FROM parking_violations
LIMIT 5
""").show(truncate=False)

+--------------+--------+------------------+----------+--------------+----------------+------------------+--------------------+------------+-----------------+--------------+------------------------------+-----------------+----------+-----------+-------------+-----------+
|summons_number|plate_id|registration_state|issue_date|violation_time|violation_county|violation_precinct|street_name         |vehicle_make|vehicle_body_type|violation_code|violation_description         |issue_date_parsed|issue_year|issue_month|issue_weekday|fiscal_year|
+--------------+--------+------------------+----------+--------------+----------------+------------------+--------------------+------------+-----------------+--------------+------------------------------+-----------------+----------+-----------+-------------+-----------+
|4933518622    |AHG9748 |NY                |01/13/2025|1150A         |ST              |0                 |NB CLOVE RD. VICTORY|FORD        |SUBN             |36            |PHTO SCHOOL

In [4]:
spark.sql("""
SELECT
    fiscal_year,
    COUNT(*) AS number_of_violations
FROM parking_violations
GROUP BY fiscal_year
ORDER BY fiscal_year
""").show()

[Stage 2:=================================>                        (8 + 4) / 14]

+-----------+--------------------+
|fiscal_year|number_of_violations|
+-----------+--------------------+
|       2023|            21563238|
|       2024|            16099641|
|       2025|            16557773|
+-----------+--------------------+



In [5]:
spark.sql("""
SELECT
    violation_code,
    violation_description,
    COUNT(*) AS number_of_violations
FROM parking_violations
GROUP BY violation_code, violation_description
ORDER BY number_of_violations DESC
LIMIT 20
""").show(truncate=False)

[Stage 5:====================================================>    (13 + 1) / 14]

+--------------+------------------------------+--------------------+
|violation_code|violation_description         |number_of_violations|
+--------------+------------------------------+--------------------+
|36            |PHTO SCHOOL ZN SPEED VIOLATION|18541868            |
|21            |21-No Parking (street clean)  |4626209             |
|38            |38-Failure to Dsplay Meter Rec|3782670             |
|14            |14-No Standing                |2567627             |
|5             |BUS LANE VIOLATION            |2279609             |
|7             |FAILURE TO STOP AT RED LIGHT  |2266484             |
|40            |40-Fire Hydrant               |1937147             |
|21            |No Parking Street Cleaning    |1690974             |
|71            |71A-Insp Sticker Expired (NYS)|1599822             |
|20            |20A-No Parking (Non-COM)      |1439924             |
|70            |70A-Reg. Sticker Expired (NYS)|1047906             |
|37            |37-Expired Parking

In [6]:
spark.sql("""
SELECT
    vehicle_make,
    COUNT(*) AS number_of_violations
FROM parking_violations
GROUP BY vehicle_make
ORDER BY number_of_violations DESC
LIMIT 20
""").show(truncate=False)

[Stage 8:============================================>            (11 + 3) / 14]

+------------+--------------------+
|vehicle_make|number_of_violations|
+------------+--------------------+
|HONDA       |6417854             |
|TOYOT       |6374836             |
|FORD        |5033828             |
|NISSA       |4253147             |
|CHEVR       |2897207             |
|ME/BE       |2802534             |
|BMW         |2678242             |
|JEEP        |2485249             |
|HYUND       |1866357             |
|LEXUS       |1345926             |
|FRUEH       |1209613             |
|ACURA       |1194584             |
|SUBAR       |1173960             |
|KIA         |1120991             |
|DODGE       |1076044             |
|AUDI        |1029631             |
|MAZDA       |996499              |
|VOLKS       |995872              |
|RAM         |822845              |
|INFIN       |812106              |
+------------+--------------------+



In [7]:
spark.sql("""
SELECT
    fiscal_year,
    issue_month,
    COUNT(*) AS number_of_violations
FROM parking_violations
GROUP BY fiscal_year, issue_month
ORDER BY fiscal_year, issue_month
""").show(50)

[Stage 11:====================================================>   (13 + 1) / 14]

+-----------+-----------+--------------------+
|fiscal_year|issue_month|number_of_violations|
+-----------+-----------+--------------------+
|       2023|          1|             1347545|
|       2023|          2|             1280207|
|       2023|          3|             1520163|
|       2023|          4|             1388337|
|       2023|          5|             1514499|
|       2023|          6|             1653111|
|       2023|          7|             2863974|
|       2023|          8|             3256235|
|       2023|          9|             2506667|
|       2023|         10|             1520943|
|       2023|         11|             1453348|
|       2023|         12|             1258209|
|       2024|          1|             1216057|
|       2024|          2|             1251548|
|       2024|          3|             1350180|
|       2024|          4|             1305669|
|       2024|          5|             1435383|
|       2024|          6|             1321606|
|       2024|

## Ergebnis

Die bereinigten Parquet-Daten konnten erfolgreich aus dem HDFS gelesen und als Spark SQL View `parking_violations` registriert werden.

Die Testqueries zeigen:
- Die Anzahl Zeilen pro Fiskaljahr stimmt mit dem Pre-processing überein.
- `violation_code = 36` ist mit Abstand der häufigste Violation Code.
- Die häufigsten Vehicle Makes sind unter anderem HONDA, TOYOT, FORD und NISSA.
- Die Monatsverteilung kann pro Fiskaljahr mit SQL abgefragt werden.

Auffällig ist, dass einzelne `violation_code`-Werte unterschiedliche Beschreibungen haben können. Für die finale Analyse sollte deshalb primär nach `violation_code` gruppiert werden.